# PCA Portfolios in Risk Regimes
**Saverio Lauriola — research notebook**

This notebook studies whether **principal-component portfolios built on a US big-tech universe** exhibit useful **regime dependence** and whether that regime sensitivity survives basic implementation frictions.

It includes:
- correlation-PCA factor-mimicking portfolios on a US tech universe;
- a single implied-volatility-based global state variable with **non-leaky expanding-quantile** regime labels;
- **static** and **walk-forward** PCA evaluation;
- **stationary block bootstrap** (Politis–Romano) for uncertainty bands on performance metrics;
- **historical risk-free adjustment** (via `^IRX`) for excess-return metrics;
- a first implementation-aware layer using **Corwin--Schultz spread-based costs**.

**Main message.** Across the current sample, **PC1 emerges as the most robust buy-and-hold exposure**; higher PCs can look attractive in some regimes, but they are generally less stable and more implementation-sensitive.

**How to use.** Run the notebook top-to-bottom. The main research pipeline ends with the walk-forward section; the final section provides optional break-even and turnover diagnostics.


## Setup and reusable helpers


In [ ]:
# -------------------------
# PARAMETERS
# -------------------------
from yfinance import download
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


plt.rcParams["figure.figsize"] = (10, 4)
pd.options.display.float_format = "{:,.6f}".format

# Example single economy universe (US Tech)
TICKERS_US_TECH = ["AAPL","MSFT","GOOGL","AMZN","META","CSCO","NVDA","ORCL","PLTR","SAP","IBM","CRM","SHOP","UBER"]

# Global regime variable (use ONE series consistently across all economies)
IV_TICKER_PRIMARY = "^VXN"   # Nasdaq-100 implied vol
IV_TICKER_FALLBACK = "^VIX"  # fallback

# Historical risk-free proxy
RF_TICKER_PRIMARY = "^IRX"   # 13-week Treasury bill annualized yield proxy (Yahoo Finance)
RF_ANN = 252                 # trading-day annualization used for daily return conversion

START = "2015-01-01"
END   = "2026-01-31"

TRAIN_END_DEFAULT = "2022-12-31"
TEST_START_DEFAULT = "2023-01-01"

# Regime definition
REGIME_Q = (0.33, 0.66)     # low/mid/high cutoffs
REGIME_FREQ = "D"           # "D" or "W" (weekly)
SMOOTH_WINDOW = 5           # e.g. 5 trading days smoothing for log IV
EWMA_LAMBDA = 0.94          # for z-score of log IV (optional)

# PCA settings
K_BAR = 7                   # number of PCs to build portfolios for

# Portfolio normalisation
GROSS_NORM = 1.0            # enforce sum(abs(w)) = 1.0

# Walk-forward settings (optional)
WF_TRAIN_YEARS = 5          # rolling training window length
WF_STEP_MONTHS = 3          # rebalance / re-estimation frequency
WF_K_CHOICE = 7             # PCs in walk-forward (keep fixed for simplicity)

# Transaction cost model (used only in "tradable strategy" overlays)
TC_BPS = 10                 # bps per 1.0 turnover (gross)


In [ ]:
# ============================================================
# HELPERS (keep everything reusable here)
# ============================================================

def linear_returns_from_prices(adj_close: pd.DataFrame) -> pd.DataFrame:
    return adj_close.pct_change().dropna(how="all")

def ewma_mean_std(x: pd.Series, lam: float = 0.94) -> pd.DataFrame:
    x = x.dropna().astype(float)
    m = np.zeros(len(x))
    v = np.zeros(len(x))
    m[0] = x.iloc[0]
    v[0] = 0.0
    for t in range(1, len(x)):
        m[t] = lam * m[t-1] + (1 - lam) * x.iloc[t]
        innov = x.iloc[t] - m[t-1]
        v[t] = lam * v[t-1] + (1 - lam) * innov**2
    s = np.sqrt(np.maximum(v, 1e-12))
    return pd.DataFrame({"ewma_mean": m, "ewma_std": s}, index=x.index)

def build_state_z_from_iv(iv_level: pd.Series,
                          freq: str = "D",
                          smooth_window: int = 5,
                          lam: float = 0.94,
                          use_zscore: bool = True) -> pd.Series:
    # Build z_t from implied vol level:
    #   x_t = log(IV_t)
    #   smooth(x_t) = rolling mean over past 'smooth_window' periods (incl. today)
    #   z_t = (smooth - ewma_mean) / ewma_std   (optional)
    iv = iv_level.dropna().astype(float)
    if freq.upper().startswith("W"):
        iv = iv.resample("W-FRI").last()  # align to week close
    x = np.log(iv)
    x_smooth = x.rolling(window=smooth_window, min_periods=smooth_window).mean().dropna()

    if not use_zscore:
        z = x_smooth.copy()
        z.name = "state_z"
        return z

    stats = ewma_mean_std(x_smooth, lam=lam)
    z = (x_smooth.loc[stats.index] - stats["ewma_mean"]) / stats["ewma_std"]
    z.name = "state_z"
    return z

def regime_labels_from_state(z: pd.Series,
                             q=(0.33, 0.66),
                             expanding: bool = True,
                             min_obs: int = 252) -> pd.Series:
    # Non-leaky LOW/MID/HIGH labels from z_t via expanding quantiles.
    z = z.dropna().astype(float)
    labels = pd.Series(index=z.index, dtype="object")

    q1, q2 = q
    if not expanding:
        a, b = z.quantile(q1), z.quantile(q2)  # leaky benchmark
        labels[z <= a] = "LOW"
        labels[(z > a) & (z <= b)] = "MID"
        labels[z > b] = "HIGH"
        return labels

    for i in range(len(z)):
        if i + 1 < min_obs:
            continue
        hist = z.iloc[: i + 1]
        a, b = hist.quantile(q1), hist.quantile(q2)
        zi = z.iloc[i]
        if zi <= a:
            labels.iloc[i] = "LOW"
        elif zi <= b:
            labels.iloc[i] = "MID"
        else:
            labels.iloc[i] = "HIGH"
    return labels.dropna()

def weighted_mean_cov(X: pd.DataFrame, p: pd.Series | None = None):
    # Mean/cov with optional probabilities p (must sum to 1 on index).
    if p is None:
        mu = X.mean(axis=0).values
        Xc = X.values - mu.reshape(1, -1)
        Sigma = (Xc.T @ Xc) / X.shape[0]
        return pd.Series(mu, index=X.columns, name="mu"), pd.DataFrame(Sigma, index=X.columns, columns=X.columns)

    X = X.loc[p.index].copy()
    w = p.values.reshape(-1, 1)
    mu = (w * X.values).sum(axis=0)
    Xc = X.values - mu.reshape(1, -1)
    Sigma = (w * Xc).T @ Xc
    return pd.Series(mu, index=X.columns, name="mu"), pd.DataFrame(Sigma, index=X.columns, columns=X.columns)

def correlation_pca(Sigma: np.ndarray):
    # Correlation PCA: rho = D^{-1} Sigma D^{-1}.
    Sigma = np.asarray(Sigma, dtype=float)
    vol = np.sqrt(np.clip(np.diag(Sigma), 1e-18, None))
    Dinv = np.diag(1.0 / vol)
    rho = Dinv @ Sigma @ Dinv
    lam, V = np.linalg.eigh(rho)   # ascending
    lam = lam[::-1]
    V = V[:, ::-1]

    # Sign convention: largest abs element in each eigenvector positive
    max_abs_row = np.argmax(np.abs(V), axis=0)
    s = np.sign(V[max_abs_row, np.arange(V.shape[0])])
    s[s == 0] = 1.0
    V = V * s.reshape(1, -1)
    return lam, V, vol, rho

def pc_portfolio_weights_from_corr_pca(V: np.ndarray, vol: np.ndarray, gross_norm: float = 1.0) -> pd.DataFrame:
    # Investible PC portfolios: w_k ∝ D^{-1} v_k, then normalize to sum(abs(w))=gross_norm.
    V = np.asarray(V, dtype=float)
    vol = np.asarray(vol, dtype=float)
    Dinv = 1.0 / vol
    W = V * Dinv.reshape(-1, 1)
    gross = np.sum(np.abs(W), axis=0)
    gross[gross == 0] = 1.0
    W = W / gross.reshape(1, -1) * gross_norm
    return pd.DataFrame(W)

def factor_returns(R: pd.DataFrame, W: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(R.values @ W.values, index=R.index, columns=[f"PC{i+1}" for i in range(W.shape[1])])

# ---------- Basic utilities ----------
def max_drawdown_from_wealth(wealth: pd.Series) -> float:
    wealth = wealth.dropna().astype(float)
    peak = wealth.cummax()
    dd = wealth/peak - 1.0
    return float(dd.min()) if len(dd) else np.nan

def ulcer_index(r: pd.Series) -> float:
    r = r.dropna().astype(float)
    if len(r)==0:
        return np.nan
    wealth = (1.0 + r).cumprod()
    peak = wealth.cummax()
    dd = wealth/peak - 1.0
    return float(np.sqrt(np.mean(dd**2)))

def cagr(r: pd.Series, ann: int = 252) -> float:
    r = r.dropna().astype(float)
    if len(r)==0:
        return np.nan
    wealth = (1.0 + r).cumprod()
    years = len(r)/ann
    return float(wealth.iloc[-1]**(1/years)-1) if years>0 else np.nan

def annualized_yield_pct_to_daily_return(y_ann_pct: pd.Series, ann: int = 252) -> pd.Series:
    """
    Convert an annualized yield quoted in percent (e.g. Yahoo ^IRX) into
    an approximate daily simple return using trading-day compounding.
    """
    y = pd.Series(y_ann_pct).astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    y = (y / 100.0).clip(lower=-0.999999)
    rf_daily = (1.0 + y) ** (1.0 / ann) - 1.0
    rf_daily.name = "rf_daily"
    return rf_daily

def _coerce_rf_daily(r: pd.Series, rf_daily: float | pd.Series | None = 0.0) -> pd.Series:
    if rf_daily is None:
        return pd.Series(0.0, index=r.index, dtype=float, name="rf_daily")
    if np.isscalar(rf_daily):
        return pd.Series(float(rf_daily), index=r.index, dtype=float, name="rf_daily")
    rf = pd.Series(rf_daily, dtype=float).reindex(r.index).ffill().fillna(0.0)
    rf.name = getattr(rf_daily, "name", "rf_daily")
    return rf

# ---------- Performance metrics (historical RF-aware for Sharpe / Sortino / Martin) ----------
def perf_metrics(r: pd.Series, rf_daily: float | pd.Series | None = 0.0, ann: int = 252) -> dict:
    r = r.dropna().astype(float)
    if len(r) < 5:
        return {"n": int(len(r))}

    rf_s = _coerce_rf_daily(r, rf_daily)
    ex = r - rf_s

    mu_d = ex.mean()
    vol_d = ex.std(ddof=1)
    shr = (mu_d/vol_d)*np.sqrt(ann) if vol_d>0 else np.nan

    downside = ex.copy()
    downside[downside > 0] = 0.0
    dvol = downside.std(ddof=1)
    sor = (mu_d/dvol)*np.sqrt(ann) if dvol>0 else np.nan

    cagr_ = cagr(r, ann=ann)
    cagr_excess = cagr(ex, ann=ann) if (1.0 + ex).min() > 0 else np.nan
    wealth = (1.0 + r).cumprod()
    mxdd = max_drawdown_from_wealth(wealth)
    calmar = (cagr_/abs(mxdd)) if (mxdd < 0 and np.isfinite(mxdd)) else np.nan

    ui = ulcer_index(r)
    rf_cagr = cagr(rf_s, ann=ann)
    martin = ((cagr_ - rf_cagr)/ui) if (ui>0 and np.isfinite(ui)) else np.nan

    return {
        "n": int(len(r)),
        "CAGR": float(cagr_),
        "CAGR_excess": float(cagr_excess) if np.isfinite(cagr_excess) else np.nan,
        "RF_CAGR": float(rf_cagr) if np.isfinite(rf_cagr) else np.nan,
        "MxDD": float(mxdd),
        "ShR": float(shr),
        "SoR": float(sor),
        "Calmar": float(calmar),
        "Ulcer": float(ui),
        "Martin": float(martin),
    }

# ---------- Regimes ----------
def regime_from_state(z: pd.Series, q_low: float = 0.33, q_high: float = 0.66) -> pd.Series:
    z = z.dropna().astype(float)
    lo = z.quantile(q_low)
    hi = z.quantile(q_high)
    reg = pd.Series(index=z.index, dtype="object")
    reg[z <= lo] = "LOW"
    reg[(z > lo) & (z < hi)] = "MID"
    reg[z >= hi] = "HIGH"
    return reg

# ---------- Metrics tables (overall + conditional) ----------
def compute_metrics_by_regime(F: pd.DataFrame, reg: pd.Series, rf_daily: float | pd.Series | None = 0.0, ann: int = 252) -> dict:
    reg = reg.reindex(F.index).dropna()
    F = F.loc[reg.index]
    out = {"overall": {c: perf_metrics(F[c], rf_daily=rf_daily, ann=ann) for c in F.columns}}
    for rn in ["LOW","MID","HIGH"]:
        idx = reg[reg==rn].index
        out[rn] = {c: perf_metrics(F.loc[idx, c], rf_daily=rf_daily, ann=ann) for c in F.columns}
    return out

def table_from_metrics(mdict: dict, key: str) -> pd.DataFrame:
    rows=[]
    for regime_name, d in mdict.items():
        for pc, mets in d.items():
            rows.append({"regime": regime_name, "PC": pc, key: mets.get(key, np.nan)})
    return pd.DataFrame(rows).pivot(index="PC", columns="regime", values=key)

# ---------- Turnover / similarity ----------
def turnover(w_prev: np.ndarray, w_now: np.ndarray) -> float:
    w_prev = np.asarray(w_prev, float).ravel()
    w_now  = np.asarray(w_now, float).ravel()
    return float(np.sum(np.abs(w_now - w_prev)))

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, float).ravel()
    b = np.asarray(b, float).ravel()
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    if na==0 or nb==0:
        return np.nan
    return float(np.dot(a,b)/(na*nb))

def align_signs_to_previous(W_prev: np.ndarray, W_now: np.ndarray) -> np.ndarray:
    # column-wise sign alignment using cosine similarity
    W_now = W_now.copy()
    for j in range(W_now.shape[1]):
        if cosine_sim(W_prev[:,j], W_now[:,j]) < 0:
            W_now[:,j] *= -1
    return W_now

def stability_from_W_hist(W_hist: list, pcs: list) -> pd.DataFrame:
    if len(W_hist) < 2:
        return pd.DataFrame()
    rows=[]
    for i in range(1, len(W_hist)):
        W_prev = W_hist[i-1]["W"][pcs].values
        W_now  = W_hist[i]["W"][pcs].values
        W_now  = align_signs_to_previous(W_prev, W_now)
        row={"rebalance": W_hist[i]["rebalance"]}
        for j,pc in enumerate(pcs):
            row[f"cos_{pc}"] = cosine_sim(W_prev[:,j], W_now[:,j])
            row[f"to_{pc}"]  = turnover(W_prev[:,j], W_now[:,j])
        rows.append(row)
    return pd.DataFrame(rows).set_index("rebalance")

# ---------- Politis–Romano stationary bootstrap ----------
def stationary_bootstrap_indices(T: int, avg_block_len: float, rng: np.random.Generator) -> np.ndarray:
    # Politis & Romano: block length geometric with mean avg_block_len
    p = 1.0/avg_block_len
    idx = np.empty(T, dtype=int)
    idx[0] = rng.integers(0, T)
    for t in range(1, T):
        if rng.random() < p:
            idx[t] = rng.integers(0, T)
        else:
            idx[t] = (idx[t-1] + 1) % T
    return idx

def stationary_bootstrap_df(df: pd.DataFrame, n_boot: int = 1000, avg_block_len: float = 10.0, seed: int = 1):
    rng = np.random.default_rng(seed)
    T = len(df)
    for _ in range(n_boot):
        ii = stationary_bootstrap_indices(T, avg_block_len, rng)
        yield df.iloc[ii].reset_index(drop=True)

def summarize_vec(x: np.ndarray) -> pd.Series:
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x)==0:
        return pd.Series({"mean": np.nan, "p05": np.nan, "p50": np.nan, "p95": np.nan, "n": 0})
    return pd.Series({
        "mean": float(np.mean(x)),
        "p05": float(np.quantile(x, 0.05)),
        "p50": float(np.quantile(x, 0.50)),
        "p95": float(np.quantile(x, 0.95)),
        "n": int(len(x))
    })

def bootstrap_metric_tables(df_joint: pd.DataFrame,
                           pc_cols: list,
                           z_col: str = "z",
                           regime_q=(0.33,0.66),
                           metrics=("CAGR","ShR","SoR","Calmar","Martin","MxDD"),
                           n_boot: int = 1000,
                           avg_block_len: float = 10.0,
                           seed: int = 1,
                           rf_daily: float | pd.Series | None = 0.0,
                           ann: int = 252) -> dict:
    # returns dict[(metric, regime)] -> table PCs x {mean,p05,p50,p95,n}
    out={}
    sims = stationary_bootstrap_df(df_joint, n_boot=n_boot, avg_block_len=avg_block_len, seed=seed)
    mats={(m,rn): np.full((n_boot, len(pc_cols)), np.nan) for m in metrics for rn in ["overall","LOW","MID","HIGH"]}
    for i, sim in enumerate(sims):
        z = sim[z_col]
        reg = regime_from_state(z, q_low=regime_q[0], q_high=regime_q[1])
        F = sim[pc_cols]
        md = compute_metrics_by_regime(F, reg, rf_daily=rf_daily, ann=ann)
        for rn in ["overall","LOW","MID","HIGH"]:
            for j,pc in enumerate(pc_cols):
                for m in metrics:
                    mats[(m,rn)][i,j] = md[rn][pc].get(m, np.nan)
    for m in metrics:
        for rn in ["overall","LOW","MID","HIGH"]:
            rows=[]
            for j,pc in enumerate(pc_cols):
                s = summarize_vec(mats[(m,rn)][:,j]); s["PC"]=pc
                rows.append(s)
            out[(m,rn)] = pd.DataFrame(rows).set_index("PC").sort_values("mean", ascending=False)
    return out


# -------------------------
# Stationary Bootstrap (Politis–Romano)
# -------------------------
def stationary_bootstrap_indices(T: int, avg_block_len: float, rng: np.random.Generator):
    p = 1.0 / avg_block_len
    idx = np.empty(T, dtype=int)
    idx[0] = rng.integers(0, T)
    for t in range(1, T):
        if rng.random() < p:
            idx[t] = rng.integers(0, T)
        else:
            idx[t] = (idx[t-1] + 1) % T
    return idx

def stationary_bootstrap_series(df: pd.DataFrame, n_boot: int = 2000, avg_block_len: float = 10.0, seed: int = 0):
    rng = np.random.default_rng(seed)
    T = len(df)
    out = []
    for _ in range(n_boot):
        idx = stationary_bootstrap_indices(T, avg_block_len, rng)
        out.append(df.iloc[idx].reset_index(drop=True))
    return out

# -------------------------
# Politis–White style automatic block length (data-driven)
# -------------------------

def _autocov(x: np.ndarray, lag: int) -> float:
    """Sample autocovariance at lag (unbiased-ish)."""
    x = np.asarray(x, float)
    n = len(x)
    if lag >= n:
        return np.nan
    x0 = x[: n - lag]
    x1 = x[lag:]
    return float(np.mean((x0 - x0.mean()) * (x1 - x1.mean())))

def _long_run_var_bartlett(x: np.ndarray, L: int) -> float:

    g0 = _autocov(x, 0)
    s = g0
    for k in range(1, L + 1):
        wk = 1.0 - k / (L + 1.0)
        gk = _autocov(x, k)
        s += 2.0 * wk * gk
    return float(max(s, 1e-18))

def politis_white_block_length(
    s: pd.Series,
    use_abs: bool = True,
    max_lag: int | None = None,
    clip: tuple[int, int] = (2, 252),
) -> int:

    x = s.dropna().astype(float).values
    if use_abs:
        x = np.abs(x)

    T = len(x)
    if T < 50:
        # Too short: return something conservative
        return int(np.clip(10, *clip))

    # Choose truncation lag for long-run variance estimation (rule of thumb)
    if max_lag is None:
        max_lag = int(np.floor(T ** (1 / 3)))
        max_lag = max(5, min(max_lag, 200))

    # Long-run variance (captures dependence)
    lrv = _long_run_var_bartlett(x, L=max_lag)
    var = float(np.var(x, ddof=1))
    var = max(var, 1e-18)

    # Heuristic Politis–White style scaling:
    # L* ∝ (lrv/var)^(2/3) * T^(1/3)
    # (constant absorbed; in practice we calibrate gently with c=1.0)
    ratio = max(lrv / var, 1e-6)
    L_star = int(np.round((ratio ** (2 / 3)) * (T ** (1 / 3))))

    return int(np.clip(L_star, *clip))

# -------------------------
# Choose block length by matching volatility clustering (ACF)
# -------------------------

def _acf(x: np.ndarray, nlags: int) -> np.ndarray:

    x = np.asarray(x, float)
    x = x - np.mean(x)
    n = len(x)
    denom = np.dot(x, x)
    if denom <= 0:
        return np.zeros(nlags + 1)
    acf = np.empty(nlags + 1, float)
    acf[0] = 1.0
    for k in range(1, nlags + 1):
        acf[k] = np.dot(x[:-k], x[k:]) / denom
    return acf

def choose_block_length_by_acf_matching(
    s: pd.Series,
    candidates: list[int] = [5, 10, 20],
    nlags: int = 20,
    n_boot: int = 300,
    use_abs: bool = True,
    seed: int = 1,
    stationary_bootstrap_indices_fn=None,
    distance: str = "l2",
) -> dict:

    if stationary_bootstrap_indices_fn is None:
        raise ValueError("Pass your helper stationary_bootstrap_indices_fn (Politis–Romano indices generator).")

    r = s.dropna().astype(float)
    x = np.abs(r.values) if use_abs else r.values
    T = len(x)
    if T < nlags + 10:
        raise ValueError("Series too short for requested nlags.")

    target = _acf(x, nlags=nlags)

    rng = np.random.default_rng(seed)
    scores = {}
    boot_means = {}

    for L in candidates:
        acfs = np.zeros((n_boot, nlags + 1))
        for b in range(n_boot):
            ii = stationary_bootstrap_indices_fn(T, float(L), rng)
            xb = x[ii]
            acfs[b, :] = _acf(xb, nlags=nlags)
        m = acfs.mean(axis=0)
        boot_means[L] = m

        diff = m[1:] - target[1:]  # ignore lag0
        if distance == "l1":
            score = float(np.mean(np.abs(diff)))
        elif distance == "l2":
            score = float(np.sqrt(np.mean(diff**2)))
        elif distance == "weighted":
            # weight shorter lags more (more relevant for clustering)
            w = 1.0 / np.arange(1, nlags + 1)
            score = float(np.sqrt(np.mean((diff**2) * w)))
        else:
            raise ValueError("distance must be 'l1', 'l2', or 'weighted'.")

        scores[L] = score

    scores_ser = pd.Series(scores).sort_index()
    best_L = int(scores_ser.idxmin())

    return {
        "best_L": best_L,
        "scores": scores_ser,
        "target_acf": target,
        "boot_acf_mean": boot_means,
        "use_abs": use_abs,
        "nlags": nlags,
        "n_boot": n_boot,
        "distance": distance
    }

# -------------------------
# Compute and table regime-based risk & performance metrics
# -------------------------

def compute_metrics_from_joint(df: pd.DataFrame, regime_q=REGIME_Q):
    z = df["z"]
    rf = df["rf"] if "rf" in df.columns else 0.0
    reg = regime_from_state(z, q_low=regime_q[0], q_high=regime_q[1])
    out = {"overall": {}, "LOW": {}, "MID": {}, "HIGH": {}}
    for pc in PCS:
        out["overall"][pc] = perf_metrics(df[pc], rf_daily=rf)
        for rn in ["LOW","MID","HIGH"]:
            idx = reg[reg == rn].index
            rf_sub = rf.loc[idx] if hasattr(rf, "loc") else rf
            out[rn][pc] = perf_metrics(df.loc[idx, pc], rf_daily=rf_sub)
    return out

def summarize_vec(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x)==0:
        return {"mean": np.nan, "p05": np.nan, "p50": np.nan, "p95": np.nan, "n": 0}
    return {"mean": float(np.mean(x)),
            "p05": float(np.quantile(x, 0.05)),
            "p50": float(np.quantile(x, 0.50)),
            "p95": float(np.quantile(x, 0.95)),
            "n": int(len(x))}

def bootstrap_ci_table(metric_key: str, regime_name: str):
    mat = np.full((N_BOOT, len(PCS)), np.nan)
    for i, sim in enumerate(sims):
        res = compute_metrics_from_joint(sim)
        for j, pc in enumerate(PCS):
            mat[i, j] = res[regime_name][pc].get(metric_key, np.nan)
    rows=[]
    for j, pc in enumerate(PCS):
        s = summarize_vec(mat[:,j]); s["PC"]=pc
        rows.append(s)
    return pd.DataFrame(rows).set_index("PC").sort_values("mean", ascending=False)

# -------------------------
# Walk-forward PCA
# -------------------------
def add_months(ts: pd.Timestamp, m: int) -> pd.Timestamp:
    return (ts + pd.offsets.DateOffset(months=m)).normalize()

def walk_forward_pca(R: pd.DataFrame,
                     start_test: str,
                     train_years: int = 5,
                     step_months: int = 3,
                     k: int = 7,
                     gross_norm: float = 1.0,
                     tc_bps: float = 0.0):
    R = R.dropna()
    t0 = pd.Timestamp(start_test)
    t_end = R.index.max()

    weights_hist = []
    f_all = []
    costs_all = []
    w_prev = None

    reb = t0
    while True:
        train_start = reb - pd.DateOffset(years=train_years)
        train_end = reb - pd.DateOffset(days=1)
        test_start = reb
        test_end = add_months(reb, step_months) - pd.DateOffset(days=1)

        if test_start > t_end:
            break

        R_train = R.loc[train_start:train_end]
        R_test = R.loc[test_start:test_end]

        if len(R_train) < 252 or len(R_test) < 5:
            reb = add_months(reb, step_months)
            continue

        mu, Sigma = weighted_mean_cov(R_train)
        lam, V, vol, rho = correlation_pca(Sigma.values)
        W = pc_portfolio_weights_from_corr_pca(V[:, :k], vol, gross_norm=gross_norm)
        W.index = R.columns
        W.columns = [f"PC{i+1}" for i in range(W.shape[1])]

        f = factor_returns(R_test, W)
        f_all.append(f)

        if tc_bps > 0.0:
            w_now = W.values
            if w_prev is None:
                c = np.zeros(W.shape[1])
            else:
                to = np.array([turnover(w_prev[:, j], w_now[:, j]) for j in range(w_now.shape[1])])
                c = (tc_bps / 1e4) * to
            seg_cost = pd.DataFrame(0.0, index=R_test.index, columns=f.columns)
            seg_cost.iloc[0, :] = c
            costs_all.append(seg_cost)
            w_prev = w_now
        else:
            w_prev = W.values

        weights_hist.append({"rebalance": reb, "train_start": train_start, "train_end": train_end, "W": W})
        reb = add_months(reb, step_months)

    F = pd.concat(f_all).sort_index() if f_all else pd.DataFrame()
    C = pd.concat(costs_all).sort_index() if costs_all else pd.DataFrame(0.0, index=F.index, columns=F.columns)
    F_net = F - C.reindex(F.index).fillna(0.0)

    return F, F_net, weights_hist


## 1) Data download & returns

In [ ]:
all_tickers = TICKERS_US_TECH + [IV_TICKER_PRIMARY, IV_TICKER_FALLBACK]
px = download(all_tickers, start=START, end=END, auto_adjust=False, progress=False).dropna(how="all")

px_adj_close_all = px["Adj Close"][TICKERS_US_TECH].copy()
px_high = px["High"][TICKERS_US_TECH].copy()
px_low = px["Low"][TICKERS_US_TECH].copy()

# For PCA we still want a common-support panel.
px_stocks = px_adj_close_all.dropna(how="any")

px_adj = px["Adj Close"].copy()
iv = px_adj[IV_TICKER_PRIMARY] if IV_TICKER_PRIMARY in px_adj.columns else None
if iv is None or iv.dropna().empty:
    iv = px_adj[IV_TICKER_FALLBACK]
iv.name = "IV"

R = linear_returns_from_prices(px_stocks)

# -------------------------
# Historical risk-free proxy
# -------------------------
rf_raw = download(RF_TICKER_PRIMARY, start=START, end=END, auto_adjust=False, progress=False)

def _extract_single_close(x):
    if isinstance(x, pd.Series):
        return x.astype(float)
    if not isinstance(x, pd.DataFrame):
        return pd.Series(dtype=float)
    for field in ["Adj Close", "Close"]:
        if field in x.columns:
            s = x[field]
            if isinstance(s, pd.DataFrame):
                s = s.iloc[:, 0]
            return s.astype(float)
    if x.shape[1] == 1:
        return x.iloc[:, 0].astype(float)
    return pd.Series(dtype=float)

rf_ann_pct = _extract_single_close(rf_raw).dropna()
if rf_ann_pct.empty:
    print(f"WARNING: no historical risk-free proxy downloaded from {RF_TICKER_PRIMARY}; using zero daily RF.")
    rf_daily_hist = pd.Series(0.0, index=R.index, name="rf_daily")
    rf_source_used = "ZERO_FALLBACK"
else:
    rf_daily_hist = annualized_yield_pct_to_daily_return(rf_ann_pct, ann=RF_ANN)
    rf_daily_hist = rf_daily_hist.reindex(R.index).ffill().fillna(0.0)
    rf_source_used = RF_TICKER_PRIMARY

print("R shape:", R.shape)
print(
    f"RF source: {rf_source_used} | "
    f"non-null daily obs on R index: {int(rf_daily_hist.notna().sum())} | "
    f"sample mean daily RF (bps): {1e4 * rf_daily_hist.mean():.4f}"
)
R.head()


## 2) Build global regime variable and non‑leaky regime labels

In [ ]:
z = build_state_z_from_iv(iv, freq=REGIME_FREQ, smooth_window=SMOOTH_WINDOW, lam=EWMA_LAMBDA, use_zscore=True)
z_daily = z.reindex(R.index).ffill().dropna()

reg = regime_labels_from_state(z_daily, q=REGIME_Q, expanding=True, min_obs=252)

print(reg.value_counts())
z_daily.loc[reg.index].plot(title="State variable z_t (aligned)")
plt.show()

tmp = pd.DataFrame({"z": z_daily.loc[reg.index], "regime": reg})
tmp["regime_code"] = tmp["regime"].map({"LOW":0,"MID":1,"HIGH":2})
tmp["regime_code"].plot(title="Regime code (0=LOW, 1=MID, 2=HIGH)")
plt.show()


## 3) Static PCA (train) → factor portfolios → conditional performance (test)

In [ ]:
train = R.loc[:TRAIN_END_DEFAULT].dropna()
test  = R.loc[TEST_START_DEFAULT:].dropna()

mu_hat, Sigma_hat = weighted_mean_cov(train)
lam, V, vol, rho = correlation_pca(Sigma_hat.values)

W = pc_portfolio_weights_from_corr_pca(V[:, :K_BAR], vol, gross_norm=GROSS_NORM)
W.index = R.columns
W.columns = [f"PC{i+1}" for i in range(W.shape[1])]

F_test = factor_returns(test, W)

reg_test = reg.reindex(F_test.index).dropna()
F_test = F_test.loc[reg_test.index]
rf_test = rf_daily_hist.reindex(F_test.index).ffill().fillna(0.0)

metrics_overall = {c: perf_metrics(F_test[c], rf_daily=rf_test) for c in F_test.columns}

def metrics_table(d: dict, key: str):
    return pd.Series({k: v.get(key, np.nan) for k,v in d.items()})

summary = pd.DataFrame({
    "CAGR": metrics_table(metrics_overall, "CAGR"),
    "CAGR_excess": metrics_table(metrics_overall, "CAGR_excess"),
    "RF_CAGR": metrics_table(metrics_overall, "RF_CAGR"),
    "ShR":  metrics_table(metrics_overall, "ShR"),
    "SoR":  metrics_table(metrics_overall, "SoR"),
    "Martin": metrics_table(metrics_overall, "Martin"),
    "MxDD": metrics_table(metrics_overall, "MxDD"),
    "Calmar": metrics_table(metrics_overall, "Calmar"),
})
print(f"Static test metrics: Sharpe / Sortino / Martin use historical RF proxy {rf_source_used}.")
summary


In [ ]:
((1+F_test.iloc[:, :7]).cumprod()).plot(title="Test Equity Curves — PC portfolios (static PCA)")
plt.show()

for rn in ["LOW","MID","HIGH"]:
    idx = reg_test[reg_test == rn].index
    cols = F_test.columns[:7]
    ((1+F_test.loc[idx, cols]).cumprod()).plot(title=f"Equity Curves within {rn} regime (subset days)")
    plt.show()


### Bootstrap block-length calibration
Use a liquid benchmark (QQQ) to choose a reasonable stationary-bootstrap block length before running the uncertainty analysis.


In [ ]:
# ============================================================
# Bootstrap block-length calibration (QQQ benchmark)
# ============================================================

px_B = download("QQQ", start=START, end=END, auto_adjust=False, progress=False)["Adj Close"].dropna(how="all")
B_ret = linear_returns_from_prices(px_B)

if isinstance(B_ret, pd.DataFrame):
    B_ret = B_ret.iloc[:, 0]
elif isinstance(B_ret, np.ndarray) and B_ret.ndim == 2 and B_ret.shape[1] == 1:
    B_ret = B_ret[:, 0]
B_ret = pd.Series(B_ret).dropna().astype(float)

res_abs = choose_block_length_by_acf_matching(
    B_ret.abs(),
    candidates=[5, 10, 15, 20, 30],
    nlags=30,
    n_boot=300,
    use_abs=True,
    seed=1,
    stationary_bootstrap_indices_fn=stationary_bootstrap_indices,
    distance="weighted",
)
print("ACF-matching best L on |QQQ returns|:", res_abs["best_L"])
display(res_abs["scores"])

L_pw = politis_white_block_length(B_ret, use_abs=True)
print("Politis–White suggested mean block length:", L_pw)


## 4) Stationary bootstrap (Politis–Romano) for uncertainty of metrics
Bootstrap jointly (PC returns + state variable) by resampling time indices, preserving dependence.


In [ ]:
# ============================================================
# Bootstrap CIs — ALL selected PCs (first 7) and key metrics
# (Politis–Romano stationary bootstrap)
# ============================================================

PCS = [c for c in F_test.columns if c.startswith("PC")][:7]
KEYS = ["CAGR", "CAGR_excess", "ShR", "SoR", "Calmar", "Martin", "MxDD"]

df_joint = pd.concat(
    [
        F_test[PCS],
        z_daily.reindex(F_test.index).rename("z"),
        rf_daily_hist.reindex(F_test.index).rename("rf"),
    ],
    axis=1,
).dropna()
print("Joint test shape:", df_joint.shape)
print(f"Bootstrap metrics use historical RF proxy {rf_source_used} for Sharpe / Sortino / Martin and excess CAGR.")

N_BOOT = 1000
AVG_BLOCK = L_pw
sims = stationary_bootstrap_series(df_joint.reset_index(drop=True), n_boot=N_BOOT, avg_block_len=AVG_BLOCK, seed=1)

for key in KEYS:
    print("\n====================", key, "====================")
    for regime_name in ["overall","LOW","MID","HIGH"]:
        display(bootstrap_ci_table(key, regime_name).head(10))


## 5) Walk-forward PCA and implementation-aware evaluation


In [ ]:
# ============================================================
# Corwin–Schultz helpers (single asset + panel + one-way cost)
# ============================================================

def corwin_schultz_spread(high: pd.Series, low: pd.Series) -> pd.Series:
    """
    Corwin–Schultz (2012) bid–ask spread estimator from daily high/low prices.
    Returns the FULL proportional spread, e.g. 0.01 = 1%.
    Input must be two aligned Series for ONE asset.
    """
    high = pd.Series(high).astype(float)
    low = pd.Series(low).astype(float)

    idx = high.dropna().index.intersection(low.dropna().index)
    high = high.loc[idx].sort_index()
    low = low.loc[idx].sort_index()

    hl = np.log(high / low)
    beta = hl.pow(2) + hl.shift(1).pow(2)

    high2 = pd.concat([high, high.shift(1)], axis=1).max(axis=1)
    low2 = pd.concat([low, low.shift(1)], axis=1).min(axis=1)
    gamma = np.log(high2 / low2).pow(2)

    k = 3 - 2 * np.sqrt(2)
    alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / k - np.sqrt(gamma / k)
    alpha = alpha.clip(lower=0)

    spread = 2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))
    spread.name = "cs_spread"
    return spread


def corwin_schultz_panel(
    px_high: pd.DataFrame,
    px_low: pd.DataFrame,
    tickers: list | None = None,
    clip_upper: float | None = 0.10,
) -> pd.DataFrame:
    """
    Apply Corwin–Schultz asset by asset on a panel of highs/lows.
    Returns a DataFrame indexed by date, columns=tickers, with FULL spreads.
    """
    px_high = px_high.copy()
    px_low = px_low.copy()

    if tickers is None:
        tickers = [c for c in px_high.columns if c in px_low.columns]

    out = {}
    for t in tickers:
        h = px_high[t].dropna()
        l = px_low[t].dropna()

        idx = h.index.intersection(l.index)
        if len(idx) < 3:
            out[t] = pd.Series(dtype=float)
            continue

        s = corwin_schultz_spread(h.loc[idx], l.loc[idx])

        if clip_upper is not None:
            s = s.clip(upper=clip_upper)

        out[t] = s

    panel = pd.DataFrame(out).sort_index()
    panel = panel.reindex(columns=tickers)
    return panel


def cs_panel_to_one_way_cost(cs_spreads: pd.DataFrame) -> pd.DataFrame:
    """
    Convert FULL spread to ONE-WAY implementation cost.
    If spread = ask-bid over mid, a one-way execution cost is half-spread.
    """
    return 0.5 * cs_spreads.astype(float)

In [ ]:
# ============================================================
# Walk-forward PCA with:
# - sign alignment vs previous rebalance
# - flat bps OR asset-level time-varying costs
# - cost charged on first realized return of each segment
# ============================================================

def add_months(ts: pd.Timestamp, m: int) -> pd.Timestamp:
    return (pd.Timestamp(ts) + pd.offsets.DateOffset(months=m)).normalize()


def _align_W_to_prev(W_prev: pd.DataFrame | None, W_now: pd.DataFrame) -> pd.DataFrame:
    """
    Align PC signs to previous rebalance so that sign flips do not create fake turnover.
    """
    if W_prev is None:
        return W_now.copy()

    W_aligned = W_now.copy()
    common_cols = [c for c in W_now.columns if c in W_prev.columns]

    for c in common_cols:
        a = W_prev[c].values.astype(float)
        b = W_now[c].values.astype(float)

        if np.dot(a, b) < 0:
            W_aligned[c] = -W_aligned[c]

    return W_aligned


def walk_forward_pca(
    R: pd.DataFrame,
    start_test: str,
    train_years: int = 5,
    step_months: int = 3,
    k: int = 7,
    gross_norm: float = 1.0,
    tc_bps: float | None = 0.0,
    asset_costs: pd.DataFrame | None = None,
    charge_initial_rebalance: bool = False,
):
    """
    Walk-forward PCA with either:
      1) flat linear transaction cost in bps via tc_bps
      2) asset-level time-varying one-way costs via asset_costs (date x asset)

    Cost of each PC at rebalance:
        cost_pc_t = sum_i one_way_cost_{i,t} * |Δw_{i,t}|

    Cost is charged on the first realized return in each out-of-sample segment.
    """

    R = R.dropna().sort_index().copy()
    t0 = pd.Timestamp(start_test)
    t_end = R.index.max()

    if asset_costs is not None:
        asset_costs = (
            asset_costs.reindex(index=R.index, columns=R.columns)
            .sort_index()
            .ffill()
            .fillna(0.0)
            .astype(float)
        )

    weights_hist = []
    f_all = []
    costs_all = []
    W_prev = None

    reb = t0
    while True:
        train_start = reb - pd.DateOffset(years=train_years)
        train_end = reb - pd.DateOffset(days=1)
        test_start = reb
        test_end = add_months(reb, step_months) - pd.DateOffset(days=1)

        if test_start > t_end:
            break

        R_train = R.loc[train_start:train_end]
        R_test = R.loc[test_start:test_end]

        if len(R_train) < 252 or len(R_test) < 5:
            reb = add_months(reb, step_months)
            continue

        mu, Sigma = weighted_mean_cov(R_train)
        lam, V, vol, rho = correlation_pca(Sigma.values)

        W = pc_portfolio_weights_from_corr_pca(V[:, :k], vol, gross_norm=gross_norm)
        W.index = R.columns
        W.columns = [f"PC{i+1}" for i in range(W.shape[1])]

        # Critical: align signs to previous rebalance BEFORE turnover/costs/returns comparison
        W = _align_W_to_prev(W_prev, W)

        f = factor_returns(R_test, W)
        f_all.append(f)

        seg_cost = pd.DataFrame(0.0, index=R_test.index, columns=f.columns)

        if charge_initial_rebalance or (W_prev is not None):
            dW = W.abs() if W_prev is None else (W - W_prev).abs()

            # first ACTUAL trading day in the segment
            trade_date = R_test.index[0]

            if asset_costs is not None:
                c_assets = asset_costs.loc[trade_date, R.columns]
                c_pc = dW.mul(c_assets, axis=0).sum(axis=0)
            else:
                c = 0.0 if tc_bps is None else float(tc_bps) / 1e4
                c_pc = dW.sum(axis=0) * c

            seg_cost.iloc[0, :] = c_pc.reindex(f.columns).values

        costs_all.append(seg_cost)
        W_prev = W.copy()

        weights_hist.append(
            {
                "rebalance": reb,
                "train_start": train_start,
                "train_end": train_end,
                "W": W.copy(),
            }
        )

        reb = add_months(reb, step_months)

    F = pd.concat(f_all).sort_index() if f_all else pd.DataFrame()
    C = pd.concat(costs_all).sort_index() if costs_all else pd.DataFrame(0.0, index=F.index, columns=F.columns)

    C = C.reindex(index=F.index, columns=F.columns).fillna(0.0)
    F_net = F - C

    return F, F_net, weights_hist

In [ ]:
# ============================================================
# Time-varying transaction costs from Corwin–Schultz spreads
# ============================================================

# Full bid-ask spreads estimated asset by asset
cs_spreads = corwin_schultz_panel(
    px_high=px_high,
    px_low=px_low,
    tickers=list(R.columns),
    clip_upper=None,
)

# Convert full spread to one-way implementation cost
asset_costs = (
    cs_panel_to_one_way_cost(cs_spreads)
    .reindex(index=R.index, columns=R.columns)
    .ffill()
    .fillna(0.0)
)

print("One-way cost summary (bps) by asset:")
display((1e4 * asset_costs).describe().T[["mean", "50%", "max"]].sort_values("50%"))

F_wf, F_wf_net, W_hist = walk_forward_pca(
    R=R,
    start_test=TEST_START_DEFAULT,
    train_years=WF_TRAIN_YEARS,
    step_months=WF_STEP_MONTHS,
    k=WF_K_CHOICE,
    gross_norm=GROSS_NORM,
    tc_bps=None,
    asset_costs=asset_costs,
    charge_initial_rebalance=False,
)

print("WF factor returns shape:", F_wf.shape, " | net:", F_wf_net.shape)

PCS_WF = [c for c in F_wf.columns if c.startswith("PC")][:7]

if not F_wf.empty:
    ((1 + F_wf[PCS_WF]).cumprod()).plot(title="Walk-forward equity curves (gross, before costs)")
    plt.show()

    ((1 + F_wf_net[PCS_WF]).cumprod()).plot(
        title="Walk-forward equity curves (net of Corwin–Schultz one-way costs)"
    )
    plt.show()

    reg_wf = reg.reindex(F_wf.index)
    rf_wf = rf_daily_hist.reindex(F_wf.index).ffill().fillna(0.0)

    m_wf_g = {"overall": {c: perf_metrics(F_wf[c], rf_daily=rf_wf) for c in PCS_WF}}
    m_wf_n = {"overall": {c: perf_metrics(F_wf_net[c], rf_daily=rf_wf) for c in PCS_WF}}

    for rn in ["LOW", "MID", "HIGH"]:
        idx = reg_wf[reg_wf == rn].index
        m_wf_g[rn] = {c: perf_metrics(F_wf.loc[idx, c], rf_daily=rf_wf) for c in PCS_WF}
        m_wf_n[rn] = {c: perf_metrics(F_wf_net.loc[idx, c], rf_daily=rf_wf) for c in PCS_WF}

    print(f"Walk-forward metrics: Sharpe / Sortino / Martin use historical RF proxy {rf_source_used}.")
    for key in ["CAGR", "CAGR_excess", "ShR", "SoR", "Calmar", "Martin", "MxDD"]:
        print("\n---", key, "(gross) ---")
        display(table_from_metrics(m_wf_g, key))

        print("---", key, "(net of CS costs) ---")
        display(table_from_metrics(m_wf_n, key))


## 6) Optional appendix — turnover, break-even, and cost diagnostics

These helper utilities are **not required** for the core pipeline above. They provide optional diagnostics for:

1. turnover-aware net performance under stylized flat costs;
2. break-even cost analysis versus a benchmark (for example, PC7 versus PC1);
3. simple performance-versus-turnover visualizations.

The definitions below are intentionally modular so they can be reused in follow-up experiments.


In [ ]:
# ============================================================
# Optional appendix helpers: flat-cost overlays and break-even analysis
# ============================================================

def metric_from_series(
    r: pd.Series,
    metric_key: str,
    rf_daily: float | pd.Series | None = 0.0,
    ann: int = 252,
):
    return perf_metrics(r, rf_daily=rf_daily, ann=ann).get(metric_key, np.nan)


def apply_transaction_costs_series(
    r: pd.Series,
    turnover: pd.Series,
    cost_bps: float,
) -> pd.Series:
    """Apply a flat linear transaction-cost proxy to a return series."""
    r = r.astype(float).copy()
    turnover = turnover.astype(float).reindex(r.index).fillna(0.0)
    c = cost_bps / 1e4
    net = r - c * turnover
    net.name = f"{r.name}_net_{cost_bps:.1f}bps" if r.name is not None else None
    return net


def apply_transaction_costs_df(
    R: pd.DataFrame,
    turnover_df: pd.DataFrame,
    cost_bps: float,
) -> pd.DataFrame:
    """Apply a flat linear transaction-cost proxy column by column."""
    R = R.astype(float).copy()
    T = turnover_df.astype(float).reindex(index=R.index, columns=R.columns).fillna(0.0)
    c = cost_bps / 1e4
    return R - c * T


def turnover_from_W_hist(W_hist: list, pcs: list[str] | None = None) -> pd.DataFrame:
    """Rebalance-to-rebalance turnover inferred from walk-forward weights."""
    records = []
    prev_W = None
    for d in W_hist:
        reb = pd.Timestamp(d["rebalance"])
        W = d["W"].copy()
        if pcs is not None:
            W = W[pcs]
        if prev_W is None:
            to = pd.Series(0.0, index=W.columns)
        else:
            common = prev_W.index.intersection(W.index)
            to = pd.Series(index=W.columns, dtype=float)
            for c in W.columns:
                w_prev = prev_W.loc[common, c].values if c in prev_W.columns else np.zeros(len(common))
                w_now = W.loc[common, c].values
                to[c] = turnover(w_prev, w_now)
        row = pd.DataFrame([to], index=[reb])
        records.append(row)
        prev_W = W
    return pd.concat(records).sort_index() if records else pd.DataFrame()


def expand_turnover_to_daily(F: pd.DataFrame, turn_reb: pd.DataFrame) -> pd.DataFrame:
    """Map rebalance turnover to the first realized daily return after each rebalance."""
    out = pd.DataFrame(0.0, index=F.index, columns=F.columns)
    if turn_reb is None or turn_reb.empty:
        return out
    for reb in turn_reb.index:
        ix = F.index[F.index >= reb]
        if len(ix) == 0:
            continue
        out.loc[ix[0], turn_reb.columns] = turn_reb.loc[reb].reindex(out.columns).fillna(0.0).values
    return out


def break_even_cost_flat(
    strategy_r: pd.Series,
    strategy_turnover: pd.Series,
    benchmark_r: pd.Series,
    metric_key: str = "Martin",
    grid_bps: np.ndarray | None = None,
    rf_daily: float | pd.Series | None = 0.0,
    ann: int = 252,
):
    """Find the flat transaction cost at which strategy and benchmark are tied."""
    if grid_bps is None:
        grid_bps = np.linspace(0, 200, 401)

    strategy_r = pd.Series(strategy_r).astype(float)
    strategy_turnover = pd.Series(strategy_turnover).astype(float).reindex(strategy_r.index).fillna(0.0)
    benchmark_r = pd.Series(benchmark_r).astype(float).reindex(strategy_r.index)

    vals = []
    for bps in grid_bps:
        net = apply_transaction_costs_series(strategy_r, strategy_turnover, cost_bps=float(bps))
        m_strat = metric_from_series(net, metric_key, rf_daily=rf_daily, ann=ann)
        m_bench = metric_from_series(benchmark_r, metric_key, rf_daily=rf_daily, ann=ann)
        vals.append((bps, m_strat, m_bench, m_strat - m_bench))
    out = pd.DataFrame(vals, columns=["bps", "metric_strategy_net", "metric_benchmark", "diff"])

    signs = np.sign(out["diff"].values)
    idx_cross = np.where(signs[:-1] * signs[1:] <= 0)[0]
    be = np.nan if len(idx_cross) == 0 else float(out.loc[idx_cross[0], "bps"])
    return {"table": out, "break_even_cost_bps": be}


def break_even_cost_by_regime(
    strategy_r: pd.Series,
    strategy_turnover: pd.Series,
    benchmark_r: pd.Series,
    z_s: pd.Series,
    q: tuple = (0.33, 0.66),
    metric_key: str = "Martin",
    grid_bps: np.ndarray | None = None,
    expanding: bool = False,
    min_obs: int = 126,
    rf_daily: float | pd.Series | None = 0.0,
    ann: int = 252,
):
    """Break-even cost analysis computed separately by regime."""
    reg = regime_from_state(z_s, q=q, expanding=expanding, min_obs=min_obs)
    idx = reg.dropna().index.intersection(pd.Series(strategy_r).dropna().index)
    reg = reg.loc[idx]

    out = {}
    for rn in ["LOW", "MID", "HIGH"]:
        ix = reg[reg == rn].index
        out[rn] = break_even_cost_flat(
            strategy_r=pd.Series(strategy_r).loc[ix],
            strategy_turnover=pd.Series(strategy_turnover).loc[ix],
            benchmark_r=pd.Series(benchmark_r).loc[ix],
            metric_key=metric_key,
            grid_bps=grid_bps,
            rf_daily=(pd.Series(rf_daily).loc[ix] if not np.isscalar(rf_daily) and rf_daily is not None else rf_daily),
            ann=ann,
        )
    return out


def plot_break_even_curve(
    strategy_r: pd.Series,
    strategy_turnover: pd.Series,
    benchmark_r: pd.Series,
    metric_key: str = "Martin",
    grid_bps: np.ndarray | None = None,
    rf_daily: float | pd.Series | None = 0.0,
    ann: int = 252,
):
    """Plot metric(strategy net of flat costs) minus metric(benchmark) across a bps grid."""
    res = break_even_cost_flat(
        strategy_r=strategy_r,
        strategy_turnover=strategy_turnover,
        benchmark_r=benchmark_r,
        metric_key=metric_key,
        grid_bps=grid_bps,
        rf_daily=rf_daily,
        ann=ann,
    )
    tb = res["table"]

    plt.figure(figsize=(8, 4))
    plt.plot(tb["bps"], tb["diff"])
    plt.axhline(0.0, linestyle="--")
    if np.isfinite(res["break_even_cost_bps"]):
        plt.axvline(res["break_even_cost_bps"], linestyle=":")
    plt.title(f"Break-even cost curve vs benchmark ({metric_key})")
    plt.xlabel("Flat transaction cost (bps)")
    plt.ylabel("Strategy metric (net) - benchmark metric")
    plt.show()
    return res


def plot_performance_turnover_frontier(
    F: pd.DataFrame,
    turnover_df: pd.DataFrame,
    pcs: list[str] | None = None,
    metric_key: str = "Martin",
    rf_daily: float | pd.Series | None = 0.0,
    ann: int = 252,
):
    """Scatter plot of average turnover versus a chosen performance metric."""
    if pcs is None:
        pcs = list(F.columns)

    rows = []
    for c in pcs:
        r = pd.Series(F[c]).dropna()
        t = pd.Series(turnover_df[c]).reindex(r.index).fillna(0.0)
        rows.append({
            "PC": c,
            "avg_turnover": float(t.mean()),
            "metric": float(metric_from_series(r, metric_key, rf_daily=rf_daily, ann=ann)),
        })
    out = pd.DataFrame(rows).set_index("PC")

    plt.figure(figsize=(7, 4))
    plt.scatter(out["avg_turnover"], out["metric"])
    for pc, row in out.iterrows():
        plt.annotate(pc, (row["avg_turnover"], row["metric"]))
    plt.title(f"{metric_key} vs average turnover")
    plt.xlabel("Average daily turnover proxy")
    plt.ylabel(metric_key)
    plt.show()
    return out


### Example usage
The following examples are optional diagnostics built on top of the walk-forward objects (`F_wf`, `W_hist`, `asset_costs`, `z_daily`).


In [ ]:
# ============================================================
# Example 2: Transaction costs on walk-forward returns
# ============================================================

# Example placeholders:
# PCS_WF = [c for c in F_wf.columns if c.startswith("PC")][:7]

# If you already computed W_hist from walk-forward PCA:
# turn_reb = turnover_from_W_hist(W_hist, pcs=PCS_WF)
# turn_daily = expand_turnover_to_daily(F_wf[PCS_WF], turn_reb)

# Apply stylized costs:
# F_wf_net_10 = apply_transaction_costs_df(F_wf[PCS_WF], turn_daily, cost_bps=10.0)
# F_wf_net_25 = apply_transaction_costs_df(F_wf[PCS_WF], turn_daily, cost_bps=25.0)

# Compare metrics:
# display(pd.DataFrame({
#     "gross_Martin": [perf_metrics(F_wf[c])["Martin"] for c in PCS_WF],
#     "net10_Martin": [perf_metrics(F_wf_net_10[c])["Martin"] for c in PCS_WF],
#     "net25_Martin": [perf_metrics(F_wf_net_25[c])["Martin"] for c in PCS_WF],
# }, index=PCS_WF))


In [ ]:
# ============================================================
# Example 3: Break-even cost analysis vs PC1
# ============================================================

# Example placeholders:
PCS_WF = [c for c in F_wf.columns if c.startswith("PC")][:7]
turn_reb = turnover_from_W_hist(W_hist, pcs=PCS_WF)
turn_daily = expand_turnover_to_daily(F_wf[PCS_WF], turn_reb)

# Example: PC7 vs PC1 on Martin ratio
res_be = plot_break_even_curve(
     strategy_r=F_wf["PC7"],
     strategy_turnover=turn_daily["PC7"],
     benchmark_r=F_wf["PC1"],
     metric_key="Martin",
)
print("Break-even cost (bps):", res_be["break_even_cost_bps"])

# Regime-conditioned break-even
# res_be_reg = break_even_cost_by_regime(
#     strategy_r=F_wf["PC7"],
#     strategy_turnover=turn_daily["PC7"],
#     benchmark_r=F_wf["PC1"],
#     z_s=z_daily.reindex(F_wf.index),
#     metric_key="Martin",
# )
# print({k: v["break_even_cost_bps"] for k, v in res_be_reg.items()})


In [ ]:
# ============================================================
# Example 4: Performance vs turnover frontier
# ============================================================

# Example placeholders:
# PCS_WF = [c for c in F_wf.columns if c.startswith("PC")][:7]
# turn_reb = turnover_from_W_hist(W_hist, pcs=PCS_WF)
# turn_daily = expand_turnover_to_daily(F_wf[PCS_WF], turn_reb)

# frontier = plot_performance_turnover_frontier(
#     F=F_wf[PCS_WF],
#     turnover_df=turn_daily[PCS_WF],
#     pcs=PCS_WF,
#     metric_key="Martin",
# )
# display(frontier.sort_values("metric", ascending=False))


## Roadmap / TODO

- refine the regime signal (for example, IV percentiles or a richer state indicator);
- test **risk-management overlays on PC1** (for example, volatility scaling or drawdown-aware de-risking);
- extend execution modelling beyond spread-based costs (slippage / impact proxies and flat-cost robustness checks);
- replicate the framework across additional regions (EU / UK / JP) with a unified regime lens;
- run broader sensitivity checks on regime definition, bootstrap block length, `k`, and covariance shrinkage.
